# Palimpsest — Reference Solution

Reconstructing edit lineages from shuffled bags of document snapshots.

**Approach.** The score is immediate-parent accuracy, and every snapshot has
exactly one parent among {other snapshots of its item} ∪ {ROOT} — so the
natural model is an **(n+1)-way classifier per snapshot**, trained with a
softmax over candidate parents *within each item*. The features for an ordered
pair (u → v) are the token-level **diff**: hashed bags of what v adds and what
v removes relative to u, plus a few numeric summaries. Direction is learnable
because human edits are asymmetric — revisions tend to add listings, prices,
links and phrasing, and to remove dead entries and typos — and a parent→child
diff is not the mirror of the child→parent diff.

Decoding respects the structure: each lineage is a chain, so each snapshot has
at most one child, which makes parent selection an **assignment problem**
(Hungarian matching, with ROOT duplicated into a handful of slots), followed by
cycle-breaking to guarantee a valid forest.

Progression shown below:
1. **v1 baseline** — tf-idf clustering + byte-size ordering (no learning)
2. **v2** — neural diff scorer, greedy decode
3. **v3 final** — same scorer, matching decode, full training budget



In [ ]:
import os, re, json, math, time, random
import torch; torch.set_num_threads(max(1, os.cpu_count() - 1))
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("./dataset/public")
WORK = Path("./working"); WORK.mkdir(exist_ok=True)

train_df = pd.read_csv(DATA / "train.csv", dtype=str, keep_default_na=False)
test_df  = pd.read_csv(DATA / "test.csv",  dtype=str, keep_default_na=False)

def load_item(item_id):
    with open(DATA / "snapshots" / f"{item_id}.json", encoding="utf8") as fh:
        return json.load(fh)["snapshots"]

train_items = sorted(train_df["item_id"].unique())
test_items  = sorted(test_df["item_id"].unique())
truth = {i: dict(zip(g["snapshot_id"], g["parent_id"]))
         for i, g in train_df.groupby("item_id")}
print(f"{len(train_items)} train items / {len(test_items)} test items, "
      f"{len(test_df)} test snapshots")


## Shared library

Featurisation (ordered-pair token diffs), the model (two `EmbeddingBag`s over
added/removed token hashes + numeric summaries, plus a small ROOT head), the
per-item softmax training loop, and both decoders.


In [ ]:
import re, math, json, random, time
from collections import Counter
import numpy as np

TOK = re.compile(r"[A-Za-z0-9_']+|[^\sA-Za-z0-9]")
DIM = 1 << 18
MAX_TOK = 128
N_NUM = 12


def tokenize(t):
    return TOK.findall(t or "")


def hidx(tok, slot):
    return (hash((slot, tok)) & 0x7FFFFFFF) % DIM


def pair_features(cu, cv, lu, lv):
    """Diff features for the ordered pair u -> v."""
    added = cv - cu
    removed = cu - cv
    na, nr = sum(added.values()), sum(removed.values())
    inter = sum((cu & cv).values())
    union = sum((cu | cv).values()) or 1
    lu, lv = max(1, lu), max(1, lv)
    num = np.array([
        math.log1p(na), math.log1p(nr),
        (na - nr) / (lu + lv),
        inter / union,
        math.log1p(lu), math.log1p(lv),
        (lv - lu) / lu,
        na / lu, nr / lu,
        1.0 if na > nr else 0.0,
        1.0 if lv > lu else 0.0,
        abs(lv - lu) / (lu + lv),
    ], dtype=np.float32)
    a_ids = [hidx(t, 0) for t, _ in added.most_common(MAX_TOK)] or [0]
    r_ids = [hidx(t, 1) for t, _ in removed.most_common(MAX_TOK)] or [0]
    return a_ids, r_ids, num


def featurise_item(sids, texts, truth=None):
    """All ordered pairs of one item, plus the per-snapshot root features."""
    toks = [tokenize(t) for t in texts]
    counts = [Counter(t) for t in toks]
    lens = [len(t) for t in toks]
    n = len(sids)
    A, R, NUM, PAIRS = [], [], [], []
    for u in range(n):
        for v in range(n):
            if u == v:
                continue
            a, r, num = pair_features(counts[u], counts[v], lens[u], lens[v])
            A.append(a); R.append(r); NUM.append(num); PAIRS.append((u, v))
    d = dict(n=n, sids=sids, pairs=PAIRS, A=A, R=R,
             NUM=np.array(NUM, dtype=np.float32),
             sizes=np.array([len(t) for t in texts], dtype=np.float32))
    if truth is not None:
        idx = {s: i for i, s in enumerate(sids)}
        d["target"] = np.array([n if truth[s] == "ROOT" else idx[truth[s]]
                                for s in sids], dtype=np.int64)   # n == ROOT class
    return d


# ------------------------------------------------------------------ model
def make_model(torch, nn, dim=128):
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.ea = nn.EmbeddingBag(DIM, dim, mode="mean")
            self.er = nn.EmbeddingBag(DIM, dim, mode="mean")
            self.num = nn.Sequential(nn.Linear(N_NUM, 96), nn.GELU(), nn.Linear(96, 96))
            self.head = nn.Sequential(
                nn.LayerNorm(dim * 2 + 96), nn.Linear(dim * 2 + 96, 384), nn.GELU(),
                nn.Dropout(0.10), nn.Linear(384, 192), nn.GELU(), nn.Linear(192, 1))
            # ROOT score: a learned function of the snapshot's own statistics
            self.root = nn.Sequential(nn.Linear(3, 64), nn.GELU(),
                                      nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1))

        def forward(self, a_in, a_off, r_in, r_off, num):
            va, vr = self.ea(a_in, a_off), self.er(r_in, r_off)
            return self.head(torch.cat([va, vr, self.num(num)], -1)).squeeze(-1)

        def root_score(self, feats):
            return self.root(feats).squeeze(-1)
    return Net()


def pack(torch, ds):
    a_in, a_off, r_in, r_off = [], [], [], []
    for d in ds:
        for a, r in zip(d["A"], d["R"]):
            a_off.append(len(a_in)); a_in += a
            r_off.append(len(r_in)); r_in += r
    num = np.vstack([d["NUM"] for d in ds])
    return (torch.tensor(a_in, dtype=torch.long), torch.tensor(a_off, dtype=torch.long),
            torch.tensor(r_in, dtype=torch.long), torch.tensor(r_off, dtype=torch.long),
            torch.tensor(num))


def root_feats(torch, d):
    """Per-snapshot features for the ROOT head: relative size and rank."""
    s = d["sizes"]
    z = (s - s.mean()) / (s.std() + 1e-6)
    rank = np.argsort(np.argsort(s)) / max(1, len(s) - 1)
    inv_n = np.full(len(s), 1.0 / len(s), dtype=np.float32)
    return torch.tensor(np.stack([z, rank, inv_n], axis=1).astype(np.float32))


def item_logits(torch, net, d, device):
    """(n, n+1) matrix: logits[v, u] = score that u is v's parent; column n is ROOT."""
    a_in, a_off, r_in, r_off, num = [t.to(device) for t in pack(torch, [d])]
    flat = net(a_in, a_off, r_in, r_off, num)
    n = d["n"]
    M = torch.full((n, n + 1), -1e4, device=device)
    pu = torch.tensor([p[0] for p in d["pairs"]], device=device)
    pv = torch.tensor([p[1] for p in d["pairs"]], device=device)
    M = M.index_put((pv, pu), flat)
    M[:, n] = net.root_score(root_feats(torch, d).to(device))
    return M


def train(data, epochs=12, lr=2e-3, dim=128, seed=0, device=None,
          batch_items=6, log=print, val=None):
    import torch, torch.nn as nn
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    dev = device or ("cuda" if torch.cuda.is_available() else
                     ("mps" if torch.backends.mps.is_available() else "cpu"))
    net = make_model(torch, nn, dim=dim).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=max(1, epochs * (len(data) // batch_items + 1)))
    lossf = nn.CrossEntropyLoss()
    t0 = time.time()
    for ep in range(epochs):
        net.train()
        order = list(range(len(data))); random.shuffle(order)
        tot = nb = 0
        for k in range(0, len(order), batch_items):
            chunk = [data[i] for i in order[k:k + batch_items]]
            opt.zero_grad()
            loss = 0.0
            for d in chunk:
                M = item_logits(torch, net, d, dev)
                loss = loss + lossf(M, torch.tensor(d["target"], device=dev))
            loss = loss / len(chunk)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
            try:
                sched.step()
            except Exception:
                pass
            tot += float(loss.detach()); nb += 1
        msg = f"  epoch {ep+1}/{epochs}  loss={tot/max(1,nb):.4f}  {time.time()-t0:.0f}s"
        if val is not None and (ep + 1) % 3 == 0:
            msg += f"  val_parent_acc={evaluate(net, val, dev):.4f}"
        log(msg)
    return net, dev


# ------------------------------------------------------------------ decode
def decode_matching(M, max_roots=6):
    """M: (n, n+1) score matrix, column n is ROOT. Returns parent index per
    snapshot (n == ROOT). Each real parent may take at most one child, so this
    is an assignment problem; ROOT is duplicated into `max_roots` columns."""
    from scipy.optimize import linear_sum_assignment
    n = M.shape[0]
    k = min(max_roots, n)
    C = np.concatenate([M[:, :n], np.tile(M[:, n:n + 1], (1, k))], axis=1)
    np.fill_diagonal(C[:, :n], -1e6)
    rows, cols = linear_sum_assignment(-C)
    parent = {}
    for r, c in zip(rows, cols):
        parent[int(r)] = int(c) if c < n else "ROOT"
    return break_cycles(parent, M, n)


def break_cycles(parent, M, n):
    """Detach the weakest edge of any cycle to ROOT."""
    for _ in range(n + 1):
        cyc = _find_cycle(parent, n)
        if not cyc:
            break
        weakest = min(cyc, key=lambda v: M[v, parent[v]] if parent[v] != "ROOT" else 1e9)
        parent[weakest] = "ROOT"
    return parent


def _find_cycle(parent, n):
    state = {}
    for start in range(n):
        path, node = [], start
        while True:
            st = state.get(node)
            if st == 1:
                break
            if st == 0:
                return path[path.index(node):]
            state[node] = 0
            path.append(node)
            nxt = parent.get(node, "ROOT")
            if nxt == "ROOT":
                break
            node = nxt
        for v in path:
            state[v] = 1
    return None


def decode_greedy(M):
    n = M.shape[0]
    parent = {}
    for v in range(n):
        row = M[v].copy()
        row[v] = -1e9
        j = int(np.argmax(row))
        parent[v] = "ROOT" if j == n else j
    return break_cycles(parent, M, n)


def predict(net, data, device, decoder=decode_matching):
    import torch
    net.eval()
    out = {}
    with torch.no_grad():
        for d in data:
            M = item_logits(torch, net, d, device).cpu().numpy().astype(np.float64)
            par = decoder(M)
            out[d["item_id"]] = {
                d["sids"][v]: ("ROOT" if p == "ROOT" else d["sids"][p])
                for v, p in par.items()}
    return out


def evaluate(net, data, device, decoder=decode_matching):
    preds = predict(net, data, device, decoder)
    ok = tot = 0
    for d in data:
        p = preds[d["item_id"]]
        for i, sid in enumerate(d["sids"]):
            tgt = d["target"][i]
            true = "ROOT" if tgt == d["n"] else d["sids"][tgt]
            ok += int(p[sid] == true); tot += 1
    return ok / max(1, tot)



## Featurise all items

Ordered pairs are quadratic in item size (an item has 12–40 snapshots), so this
is the CPU-heavy step; everything after runs on the GPU.


In [ ]:
t0 = time.time()

def featurise_split(item_ids, labelled):
    out = []
    for k, iid in enumerate(item_ids):
        snaps = load_item(iid)
        sids = sorted(snaps)
        d = featurise_item(sids, [snaps[s] for s in sids],
                           truth=truth[iid] if labelled else None)
        d["item_id"] = iid
        out.append(d)
        if (k + 1) % 100 == 0:
            print(f"  {k+1}/{len(item_ids)}  {time.time()-t0:.0f}s", flush=True)
    return out

data_train = featurise_split(train_items, labelled=True)
data_test  = featurise_split(test_items,  labelled=False)
print(f"featurised in {time.time()-t0:.0f}s")

# item-level validation split (rubric: never split snapshots within an item)
rng = random.Random(0)
idx = list(range(len(data_train))); rng.shuffle(idx)
n_val = max(1, len(idx) // 6)
data_val = [data_train[i] for i in idx[:n_val]]
data_fit = [data_train[i] for i in idx[n_val:]]
print(f"fit {len(data_fit)} items / val {len(data_val)} items")


## v1 — no-learning baseline: cluster, then order by byte size

Group snapshots by tf-idf similarity (the number of documents is unknown, so
pick k by silhouette over k ∈ {2,3,4}), then chain each cluster smallest→largest.
This is the anchor a learned model has to beat: windows were selected for churn,
so byte size orders a lineage at barely better than a coin flip.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

def v1_predict(d, texts):
    v = TfidfVectorizer(max_features=40000, token_pattern=r"[^\s]+")
    X = v.fit_transform(texts).toarray().astype(np.float32)
    X /= np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)
    S = X @ X.T
    D = np.maximum(1.0 - S, 0); np.fill_diagonal(D, 0)
    best_k, best_lab, best_sc = 2, None, -2
    for k in range(2, 5):
        if k >= len(texts):
            break
        lab = AgglomerativeClustering(n_clusters=k, metric="precomputed",
                                      linkage="average").fit_predict(D)
        try:
            sc = silhouette_score(D, lab, metric="precomputed")
        except ValueError:
            sc = -1
        if sc > best_sc:
            best_k, best_lab, best_sc = k, lab, sc
    sizes = [len(t) for t in texts]
    pred = {}
    for c in set(best_lab):
        members = sorted((i for i in range(len(texts)) if best_lab[i] == c),
                         key=lambda i: sizes[i])
        for j, i in enumerate(members):
            pred[d["sids"][i]] = "ROOT" if j == 0 else d["sids"][members[j - 1]]
    return pred

def score_pred(pred, d):
    ok = 0
    for i, sid in enumerate(d["sids"]):
        true = "ROOT" if d["target"][i] == d["n"] else d["sids"][d["target"][i]]
        ok += int(pred.get(sid) == true)
    return ok / d["n"]

v1_val = float(np.mean([score_pred(v1_predict(d, [load_item(d["item_id"])[s] for s in d["sids"]]), d)
                        for d in data_val]))
print(f"v1 (cluster + size order)  val parent_acc = {v1_val:.4f}")


## v2 — neural diff scorer, greedy decode

Short training run to show the learned direction signal on its own; greedy
per-snapshot argmax with cycle-breaking.


In [ ]:
net2, dev = train(data_fit, epochs=6, val=data_val, seed=0, device="cpu")
v2_val = evaluate(net2, data_val, dev, decoder=decode_greedy)
print(f"v2 (neural, greedy)        val parent_acc = {v2_val:.4f}")


## v3 — full training + matching decode (final)

Full epoch budget on all training items, decoded with the Hungarian matching
(each snapshot may parent at most one child) plus cycle-breaking.


In [ ]:
# Honest measurement of the matching decode: SAME model as v2 (which never saw
# the validation items), only the decoder changes. This isolates the decoding
# gain without contaminating the estimate.
v3_val = evaluate(net2, data_val, dev, decoder=decode_matching)
print(f"v3 (neural, matching)      val parent_acc = {v3_val:.4f}")
print(f"progression: v1 {v1_val:.4f} -> v2 {v2_val:.4f} -> v3 {v3_val:.4f}")
print(f"  matching decode over greedy: {v3_val - v2_val:+.4f}")

# Final model for the submission: refit on ALL training items. No score is
# reported for it -- every item is now in its training set, so any number would
# be contaminated. The honest expectation for the test set is v3_val above.
print("")
print("refitting on all 630 training items for the submission ...")
net3, dev = train(data_train, epochs=15, seed=0, device="cpu")


## Predict the test items and write the submission


In [ ]:
preds = predict(net3, data_test, dev, decoder=decode_matching)
rows = []
for iid, g in test_df.groupby("item_id"):
    p = preds[iid]
    for sid in g["snapshot_id"]:
        rows.append((iid, sid, p[sid]))
sub = pd.DataFrame(rows, columns=["item_id", "snapshot_id", "parent_id"])
sub.to_csv(WORK / "submission.csv", index=False)
print(f"wrote {len(sub)} rows -> working/submission.csv")

# structural self-check (same rules the grader enforces)
assert len(sub) == len(test_df)
assert not sub.duplicated(["item_id", "snapshot_id"]).any()
for iid, g in sub.groupby("item_id"):
    ids = set(g["snapshot_id"]); par = dict(zip(g["snapshot_id"], g["parent_id"]))
    assert all(p == "ROOT" or (p in ids and p != s) for s, p in par.items())
    for s in par:                       # forest check
        seen, cur = set(), s
        while par.get(cur, "ROOT") != "ROOT":
            cur = par[cur]
            assert cur not in seen, f"cycle in {iid}"
            seen.add(cur)
print("submission is structurally valid")
